In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine, text

# Database connection details
db_user = 'root'
db_password = ''
db_host = 'localhost'
db_name = 'stock'
table_name = 'price'
backup_table_name = 'price_backup'

# Create a connection to the database
# const = create_engine("mysql+pymysql://root:@localhost:3306/stock")
const = create_engine(f'mysql+pymysql://{db_user}:{db_password}@{db_host}/{db_name}')

In [2]:
# Get the user's home directory
user_path = os.path.expanduser('~')
# Get the current working directory
current_path = os.getcwd()
# Derive the base directory (base_dir) by removing the last folder ('Daily')
base_path = os.path.dirname(current_path)
#C:\Users\PC1\OneDrive\A5\Data
dts_path = os.path.join(user_path, "OneDrive", "Downloads", "Datasets")
print(f"Datasets path: {dts_path}")
dat_path = os.path.join(base_path, "Data")
#C:\Users\PC1\OneDrive\Imports\santisoontarinka@gmail.com - Google Drive\Data>
god_path = os.path.join(user_path, "OneDrive","Imports","santisoontarinka@gmail.com - Google Drive","Data")
#C:\Users\PC1\iCloudDrive\data
icd_path = os.path.join(user_path, "iCloudDrive", "Data")
#C:\Users\PC1\OneDrive\Documents\obsidian-git-sync\Data
osd_path = os.path.join(user_path, "OneDrive","Documents","obsidian-git-sync","Data")

Datasets path: C:\Users\User\OneDrive\Downloads\Datasets


In [3]:
print("User path:", user_path)
print(f"Current path: {current_path}")
print(f"Base path: {base_path}")
print(f"Datasets path : {dts_path}") 
print(f"Data path : {dat_path}") 
print(f"Google Drive path : {god_path}")
print(f"iCloudDrive path : {icd_path}") 
print(f"Obsidian path : {osd_path}")

User path: C:\Users\User
Current path: C:\Users\User\OneDrive\A4\Data Cleansing
Base path: C:\Users\User\OneDrive\A4
Datasets path : C:\Users\User\OneDrive\Downloads\Datasets
Data path : C:\Users\User\OneDrive\A4\Data
Google Drive path : C:\Users\User\OneDrive\Imports\santisoontarinka@gmail.com - Google Drive\Data
iCloudDrive path : C:\Users\User\iCloudDrive\Data
Obsidian path : C:\Users\User\OneDrive\Documents\obsidian-git-sync\Data


In [40]:
# Read the table into a Pandas DataFrame
df = pd.read_sql(f'SELECT * FROM {table_name} ORDER BY date DESC', con=const)
df.shape

(338888, 7)

In [50]:
file_name = "name-ttl.csv"
input_file = os.path.join(dat_path, file_name)
print(f"Input file: {input_file}")

Input file: C:\Users\User\OneDrive\A4\Data\name-ttl.csv


In [52]:
df_name = pd.read_csv(input_file, header=None, names=['name'])
df_name.shape

(190, 1)

In [54]:
stock_list = df['name'].unique().tolist()
stock_list_str = ("','").join(stock_list)
print(len(stock_list))

190


In [56]:
file_name = "set_symbol_name_changes.csv"
input_file = os.path.join(dat_path, file_name)
print(f"Input file: {input_file}")

Input file: C:\Users\User\OneDrive\A4\Data\set_symbol_name_changes.csv


In [58]:
df_changes = pd.read_csv(input_file )
df_changes

,effective_year,old_name,old_symbol,new_name,new_symbol,effective_date,source_ids
0,2023,JWD INFOLOGISTICS PUBLIC COMPANY LIMITED,JWD,SCGJWD LOGISTICS PUBLIC COMPANY LIMITED,SJWD,2023-02-17,web:19
1,2023,UNITED POWER OF ASIA PUBLIC COMPANY LIMITED,UPA,GREEN TECH VENTURES PUBLIC COMPANY LIMITED,GTV,2023-03-01,web:30
2,2023,SIAM MAKRO PUBLIC COMPANY LIMITED,MAKRO,CP AXTRA PUBLIC COMPANY LIMITED,CPAXT,2023-06-21,web:38
3,2024,JASMINE BROADBAND INTERNET INFRASTRUCTURE FUND,JASIF,3BB INTERNET INFRASTRUCTURE FUND,3BBIF,2024-02-27,web:32
4,2024,GIFT INFINITE PUBLIC COMPANY LIMITED,GIFT,RSXYZ PUBLIC COMPANY LIMITED,RSXYZ,2024-12-24,web:16
5,2025,WOW FACTOR PUBLIC COMPANY LIMITED,W,X BIOSCIENCE PUBLIC COMPANY LIMITED,XBIO,2025-01-13,web:18
6,2026,NEWS NETWORK CORPORATION PUBLIC COMPANY LIMITED,NEWS,SPT X PUBLIC COMPANY LIMITED,SPTX,2026-01-20,web:7


In [60]:
df_changes.dtypes

effective_year     int64
old_name          object
old_symbol        object
new_name          object
new_symbol        object
effective_date    object
source_ids        object
dtype: object

In [34]:
# --- Change Names in Price Table ---

print("\n" + "="*80)
print("CHANGE SYMBOL NAMES IN PRICE TABLE")
print("="*80)

# 1. Read the changes file with headers
file_name = "set_symbol_name_changes.csv"
input_file = os.path.join(dat_path, file_name)
print(f"Input file: {input_file}")

# Read with headers (no need to specify names)
df_changes = pd.read_csv(input_file)
print(f"\nLoaded {len(df_changes)} symbol changes:")
print(df_changes[['old_symbol', 'new_symbol', 'effective_date']])

# 2. Create backup of price table before making changes
print("\n" + "-"*60)
print("CREATING BACKUP OF PRICE TABLE")
print("-"*60)

# Create timestamp for backup
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Get a connection from the engine
conn = const.connect()

try:
    # Create backup table
    backup_table_name = f"price_backup_{timestamp}"
    
    with conn.begin():
        conn.execute(text(f"""
            CREATE TABLE {backup_table_name} AS 
            SELECT * FROM price
        """))
        print(f"✓ Backup table created: {backup_table_name}")
        
        # Get backup count
        result = conn.execute(text(f"SELECT COUNT(*) FROM {backup_table_name}"))
        backup_count = result.fetchone()[0]
        print(f"  - Backed up {backup_count} rows")
except Exception as e:
    print(f"Error creating backup: {e}")
    conn.rollback()
    raise

# 3. Process each symbol change
print("\n" + "-"*60)
print("PROCESSING SYMBOL CHANGES")
print("-"*60)

# Store results
change_results = []

for idx, row in df_changes.iterrows():
    old_symbol = row['old_symbol'].strip().upper()
    new_symbol = row['new_symbol'].strip().upper()
    effective_date = row['effective_date']
    
    print(f"\nProcessing: {old_symbol} → {new_symbol} (effective: {effective_date})")
    
    # Check if old symbol exists in price table
    check_sql = text("SELECT COUNT(*) FROM price WHERE name = :old_symbol")
    result = conn.execute(check_sql, {"old_symbol": old_symbol})
    old_count = result.fetchone()[0]
    
    if old_count == 0:
        print(f"  ⚠ Warning: {old_symbol} not found in price table")
        change_results.append({
            'old_symbol': old_symbol,
            'new_symbol': new_symbol,
            'effective_date': effective_date,
            'rows_changed': 0,
            'status': 'Not found'
        })
        continue
    
    # Check if new symbol already exists
    check_new_sql = text("SELECT COUNT(*) FROM price WHERE name = :new_symbol")
    result = conn.execute(check_new_sql, {"new_symbol": new_symbol})
    new_count = result.fetchone()[0]
    
    if new_count > 0:
        print(f"  ⚠ Warning: {new_symbol} already exists with {new_count} rows")
        print(f"  Would overwrite {new_count} existing rows. Skipping to avoid data loss.")
        change_results.append({
            'old_symbol': old_symbol,
            'new_symbol': new_symbol,
            'effective_date': effective_date,
            'rows_changed': 0,
            'status': f'Target exists ({new_count} rows)'
        })
        continue
    
    # Perform the update
    try:
        update_sql = text("""
            UPDATE price 
            SET name = :new_symbol 
            WHERE name = :old_symbol
        """)
        
        result = conn.execute(update_sql, {
            "new_symbol": new_symbol,
            "old_symbol": old_symbol
        })
        conn.commit()
        
        rows_changed = result.rowcount
        print(f"  ✓ Changed {rows_changed} rows")
        
        change_results.append({
            'old_symbol': old_symbol,
            'new_symbol': new_symbol,
            'effective_date': effective_date,
            'rows_changed': rows_changed,
            'status': 'Success'
        })
        
    except Exception as e:
        print(f"  ✗ Error updating: {e}")
        conn.rollback()
        change_results.append({
            'old_symbol': old_symbol,
            'new_symbol': new_symbol,
            'effective_date': effective_date,
            'rows_changed': 0,
            'status': f'Error: {str(e)}'
        })

# 4. Summary report
print("\n" + "="*80)
print("SUMMARY REPORT")
print("="*80)

# Create results DataFrame
df_results = pd.DataFrame(change_results)
print("\nChange Summary:")
print(df_results[['old_symbol', 'new_symbol', 'effective_date', 'rows_changed', 'status']].to_string(index=False))

# Statistics
total_changed = df_results[df_results['status'] == 'Success']['rows_changed'].sum()
successful_changes = len(df_results[df_results['status'] == 'Success'])
failed_changes = len(df_results[df_results['status'] != 'Success'])

print(f"\nStatistics:")
print(f"  - Total symbol changes processed: {len(df_changes)}")
print(f"  - Successful changes: {successful_changes}")
print(f"  - Failed/Skipped changes: {failed_changes}")
print(f"  - Total rows updated: {total_changed}")

# 5. Show details of successful changes
if successful_changes > 0:
    print("\n" + "-"*60)
    print("SUCCESSFUL CHANGES DETAILS")
    print("-"*60)
    
    for result in change_results:
        if result['status'] == 'Success':
            old_symbol = result['old_symbol']
            new_symbol = result['new_symbol']
            rows = result['rows_changed']
            print(f"  ✓ {old_symbol} → {new_symbol}: updated {rows} rows")

# 6. Show warnings for symbols not found
not_found = df_results[df_results['status'] == 'Not found']
if len(not_found) > 0:
    print("\n" + "-"*60)
    print("SYMBOLS NOT FOUND IN PRICE TABLE")
    print("-"*60)
    for _, row in not_found.iterrows():
        print(f"  ⚠ {row['old_symbol']} (effective: {row['effective_date']})")

# 7. Show conflicts
conflicts = df_results[df_results['status'].str.contains('Target exists', na=False)]
if len(conflicts) > 0:
    print("\n" + "-"*60)
    print("CONFLICTS - NEW SYMBOLS ALREADY EXIST")
    print("-"*60)
    for _, row in conflicts.iterrows():
        print(f"  ⚠ {row['old_symbol']} → {row['new_symbol']}: {row['status']}")

# 8. Save change report to CSV
report_file = os.path.join(dat_path, f"symbol_change_report_{timestamp}.csv")
df_results.to_csv(report_file, index=False)
print(f"\n✓ Change report saved to: {report_file}")

# 9. Option to rollback if needed
print("\n" + "-"*60)
print("ROLLBACK INFORMATION")
print("-"*60)
print(f"If you need to rollback, you can restore from backup table: {backup_table_name}")
print(f"Rollback SQL: INSERT INTO price SELECT * FROM {backup_table_name}")
print(f"Then: DROP TABLE {backup_table_name}")

# Close the connection
conn.close()

print("\n" + "="*80)
print("SYMBOL NAME CHANGES COMPLETED")
print("="*80)


CHANGE SYMBOL NAMES IN PRICE TABLE
Input file: C:\Users\User\OneDrive\A4\Data\set_symbol_name_changes.csv

Loaded 7 symbol changes:
  old_symbol new_symbol effective_date
0        JWD       SJWD     2023-02-17
1        UPA        GTV     2023-03-01
2      MAKRO      CPAXT     2023-06-21
3      JASIF      3BBIF     2024-02-27
4       GIFT      RSXYZ     2024-12-24
5          W       XBIO     2025-01-13
6       NEWS       SPTX     2026-01-20

------------------------------------------------------------
CREATING BACKUP OF PRICE TABLE
------------------------------------------------------------
✓ Backup table created: price_backup_20260327_145953
  - Backed up 389382 rows

------------------------------------------------------------
PROCESSING SYMBOL CHANGES
------------------------------------------------------------

Processing: JWD → SJWD (effective: 2023-02-17)
  ⚠ Warning: JWD not found in price table

Processing: UPA → GTV (effective: 2023-03-01)
  ⚠ Warning: UPA not found in price 

In [36]:
# --- Delete Rows from Price Table Where Name Not in df_name ---

print("\n" + "="*80)
print("DELETE ROWS FROM PRICE TABLE WHERE NAME NOT IN NAME LIST")
print("="*80)

# 1. Get unique names from price table and df_name
print("\n" + "-"*60)
print("ANALYZING DATA")
print("-"*60)

# Get all unique names from price table
price_names_set = set(df['name'].unique())
print(f"Total unique names in price table: {len(price_names_set)}")

# Get names from df_name (the allowed names)
allowed_names_set = set(df_name['name'].str.upper().str.strip())
print(f"Allowed names from df_name: {len(allowed_names_set)}")

# Find names to delete (in price but not in allowed list)
names_to_delete = price_names_set - allowed_names_set
print(f"Names to delete: {len(names_to_delete)}")

if len(names_to_delete) == 0:
    print("\n✓ No rows to delete. All price table names are in the allowed list.")
else:
    # 2. Create backup of rows to be deleted
    print("\n" + "-"*60)
    print("CREATING BACKUP OF ROWS TO BE DELETED")
    print("-"*60)
    
    # Create timestamp for backup
    from datetime import datetime
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Get all rows that will be deleted
    names_list = list(names_to_delete)
    
    # Query to get rows to delete (using parameterized query with placeholders)
    # Since we have many names, we'll process in batches to avoid query size limits
    batch_size = 100
    all_deleted_rows = []
    
    print(f"Fetching rows to delete for {len(names_list)} names...")
    
    # Get connection
    conn = const.connect()
    
    try:
        for i in range(0, len(names_list), batch_size):
            batch = names_list[i:i+batch_size]
            placeholders = ','.join(['%s'] * len(batch))
            
            sql = f"""
            SELECT * FROM price 
            WHERE name IN ({placeholders})
            """
            
            df_batch = pd.read_sql(sql, conn, params=tuple(batch))
            all_deleted_rows.append(df_batch)
            
            print(f"  - Processed batch {i//batch_size + 1}: found {len(df_batch)} rows")
        
        # Combine all batches
        if all_deleted_rows:
            df_to_delete = pd.concat(all_deleted_rows, ignore_index=True)
            print(f"\n✓ Total rows to delete: {len(df_to_delete)}")
        else:
            df_to_delete = pd.DataFrame()
            print("\n✓ No rows found to delete")
        
        # Save backup of rows to be deleted
        if len(df_to_delete) > 0:
            backup_file = os.path.join(dat_path, f"deleted_rows_backup_{timestamp}.csv")
            df_to_delete.to_csv(backup_file, index=False)
            print(f"✓ Backup saved to: {backup_file}")
            
            # Show sample of deleted rows by name
            print("\nSample of rows to delete (by name):")
            name_counts = df_to_delete['name'].value_counts().head(20)
            for name, count in name_counts.items():
                print(f"  - {name}: {count} rows")
            
            if len(names_to_delete) > 20:
                print(f"  ... and {len(names_to_delete) - 20} more names")
        
        # 3. Confirm deletion
        print("\n" + "-"*60)
        print("DELETION CONFIRMATION")
        print("-"*60)
        
        # Show statistics
        print(f"Current rows in price table: {len(df)}")
        print(f"Rows to delete: {len(df_to_delete)}")
        print(f"Rows that will remain: {len(df) - len(df_to_delete)}")
        
        # Ask for confirmation (uncomment if you want interactive confirmation)
        # confirm = input("\nDo you want to proceed with deletion? (yes/no): ")
        # if confirm.lower() != 'yes':
        #     print("Deletion cancelled.")
        #     conn.close()
        #     raise SystemExit
        
        # 4. Perform deletion
        print("\n" + "-"*60)
        print("PERFORMING DELETION")
        print("-"*60)
        
        # Delete in batches to avoid transaction issues
        deleted_count = 0
        batch_size_delete = 1000
        
        for i in range(0, len(names_list), batch_size_delete):
            batch = names_list[i:i+batch_size_delete]
            placeholders = ','.join(['%s'] * len(batch))
            
            delete_sql = text(f"""
                DELETE FROM price 
                WHERE name IN ({placeholders})
            """)
            
            result = conn.execute(delete_sql, tuple(batch))
            conn.commit()
            
            deleted_count += result.rowcount
            print(f"  - Deleted batch {i//batch_size_delete + 1}: {result.rowcount} rows")
        
        print(f"\n✓ Total rows deleted: {deleted_count}")
        
        # 5. Verification
        print("\n" + "-"*60)
        print("VERIFICATION")
        print("-"*60)
        
        # Verify deletion by checking remaining rows
        result = conn.execute(text("SELECT COUNT(*) FROM price"))
        remaining_count = result.fetchone()[0]
        
        # Check if any of the deleted names still exist
        check_sql = text(f"""
            SELECT COUNT(*) FROM price 
            WHERE name IN ({','.join(['%s'] * min(10, len(names_list)))})
        """)
        
        # Sample check for first 10 names
        sample_names = list(names_to_delete)[:10]
        if sample_names:
            result = conn.execute(check_sql, tuple(sample_names))
            remaining_deleted = result.fetchone()[0]
        else:
            remaining_deleted = 0
        
        print(f"Expected remaining rows: {len(df) - len(df_to_delete)}")
        print(f"Actual remaining rows: {remaining_count}")
        print(f"Rows that should have been deleted but remain: {remaining_deleted}")
        
        if remaining_count == len(df) - len(df_to_delete):
            print("✓ Verification successful!")
        else:
            print("⚠ Warning: Row count mismatch. Please check the backup file.")
        
        # 6. Summary report
        print("\n" + "="*80)
        print("DELETION SUMMARY")
        print("="*80)
        
        # Create summary DataFrame
        summary_data = []
        for name in list(names_to_delete)[:50]:  # Show first 50 for summary
            count = len(df_to_delete[df_to_delete['name'] == name])
            summary_data.append({'name': name, 'rows_deleted': count})
        
        df_summary = pd.DataFrame(summary_data)
        print("\nTop 50 deleted names:")
        print(df_summary.to_string(index=False))
        
        # Save summary report
        summary_file = os.path.join(dat_path, f"deletion_summary_{timestamp}.csv")
        df_summary.to_csv(summary_file, index=False)
        print(f"\n✓ Deletion summary saved to: {summary_file}")
        
        # 7. Rollback information
        print("\n" + "-"*60)
        print("ROLLBACK INFORMATION")
        print("-"*60)
        print(f"Backup file: {backup_file}")
        print(f"To restore deleted rows, you can run:")
        print(f"  df_restore = pd.read_csv('{backup_file}')")
        print(f"  df_restore.to_sql('price', con=const, if_exists='append', index=False)")
        
    except Exception as e:
        print(f"\n✗ Error during deletion: {e}")
        conn.rollback()
        print("Transaction rolled back. No changes were made.")
    
    finally:
        # Close connection
        conn.close()

print("\n" + "="*80)
print("OPERATION COMPLETED")
print("="*80)


DELETE ROWS FROM PRICE TABLE WHERE NAME NOT IN NAME LIST

------------------------------------------------------------
ANALYZING DATA
------------------------------------------------------------
Total unique names in price table: 233
Allowed names from df_name: 190
Names to delete: 43

------------------------------------------------------------
CREATING BACKUP OF ROWS TO BE DELETED
------------------------------------------------------------
Fetching rows to delete for 43 names...
  - Processed batch 1: found 50494 rows

✓ Total rows to delete: 50494
✓ Backup saved to: C:\Users\User\OneDrive\A4\Data\deleted_rows_backup_20260327_150549.csv

Sample of rows to delete (by name):
  - TIPCO: 1930 rows
  - KSL: 1809 rows
  - RS: 1783 rows
  - S11: 1779 rows
  - TK: 1778 rows
  - INTUCH: 1746 rows
  - EASTW: 1728 rows
  - ECL: 1721 rows
  - LHK: 1569 rows
  - STANLY: 1568 rows
  - PAP: 1567 rows
  - STEC: 1550 rows
  - SPRIME: 1494 rows
  - SKN: 1448 rows
  - AJ: 1442 rows
  - ESSO: 1418 row

In [38]:
# --- Delete Rows from Price Table Where Name Not in df_name ---

print("\n" + "="*80)
print("DELETE ROWS FROM PRICE TABLE WHERE NAME NOT IN NAME LIST")
print("="*80)

# 1. Get unique names from price table and df_name
print("\n" + "-"*60)
print("ANALYZING DATA")
print("-"*60)

# Get all unique names from price table
price_names_set = set(df['name'].unique())
print(f"Total unique names in price table: {len(price_names_set)}")

# Get names from df_name (the allowed names)
allowed_names_set = set(df_name['name'].str.upper().str.strip())
print(f"Allowed names from df_name: {len(allowed_names_set)}")

# Find names to delete (in price but not in allowed list)
names_to_delete = price_names_set - allowed_names_set
print(f"Names to delete: {len(names_to_delete)}")

if len(names_to_delete) == 0:
    print("\n✓ No rows to delete. All price table names are in the allowed list.")
else:
    # 2. Create backup of rows to be deleted
    print("\n" + "-"*60)
    print("CREATING BACKUP OF ROWS TO BE DELETED")
    print("-"*60)
    
    # Create timestamp for backup
    from datetime import datetime
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Get all rows that will be deleted
    names_list = list(names_to_delete)
    
    # Query to get rows to delete (using parameterized query with placeholders)
    # Since we have many names, we'll process in batches to avoid query size limits
    batch_size = 100
    all_deleted_rows = []
    
    print(f"Fetching rows to delete for {len(names_list)} names...")
    
    # Get connection
    conn = const.connect()
    
    try:
        for i in range(0, len(names_list), batch_size):
            batch = names_list[i:i+batch_size]
            # Create placeholders for the query
            placeholders = ','.join(['%s'] * len(batch))
            
            sql = f"""
            SELECT * FROM price 
            WHERE name IN ({placeholders})
            """
            
            df_batch = pd.read_sql(sql, conn, params=tuple(batch))
            all_deleted_rows.append(df_batch)
            
            print(f"  - Processed batch {i//batch_size + 1}: found {len(df_batch)} rows")
        
        # Combine all batches
        if all_deleted_rows:
            df_to_delete = pd.concat(all_deleted_rows, ignore_index=True)
            print(f"\n✓ Total rows to delete: {len(df_to_delete)}")
        else:
            df_to_delete = pd.DataFrame()
            print("\n✓ No rows found to delete")
        
        # Save backup of rows to be deleted
        if len(df_to_delete) > 0:
            backup_file = os.path.join(dat_path, f"deleted_rows_backup_{timestamp}.csv")
            df_to_delete.to_csv(backup_file, index=False)
            print(f"✓ Backup saved to: {backup_file}")
            
            # Show sample of deleted rows by name
            print("\nSample of rows to delete (by name):")
            name_counts = df_to_delete['name'].value_counts().head(20)
            for name, count in name_counts.items():
                print(f"  - {name}: {count} rows")
            
            if len(names_to_delete) > 20:
                print(f"  ... and {len(names_to_delete) - 20} more names")
        
        # 3. Confirm deletion (optional - uncomment if you want confirmation)
        print("\n" + "-"*60)
        print("DELETION CONFIRMATION")
        print("-"*60)
        
        # Show statistics
        print(f"Current rows in price table: {len(df)}")
        print(f"Rows to delete: {len(df_to_delete)}")
        print(f"Rows that will remain: {len(df) - len(df_to_delete)}")
        
        # Uncomment for interactive confirmation
        # confirm = input("\nDo you want to proceed with deletion? (yes/no): ")
        # if confirm.lower() != 'yes':
        #     print("Deletion cancelled.")
        #     conn.close()
        #     raise SystemExit
        
        # 4. Perform deletion
        print("\n" + "-"*60)
        print("PERFORMING DELETION")
        print("-"*60)
        
        # Delete in batches using a different approach
        deleted_count = 0
        batch_size_delete = 100
        
        # Method 1: Using string formatting for the IN clause (safer with MySQL)
        for i in range(0, len(names_list), batch_size_delete):
            batch = names_list[i:i+batch_size_delete]
            # Create quoted string list for SQL IN clause
            quoted_names = ["'" + name.replace("'", "''") + "'" for name in batch]
            names_in_clause = ','.join(quoted_names)
            
            delete_sql = f"""
                DELETE FROM price 
                WHERE name IN ({names_in_clause})
            """
            
            result = conn.execute(text(delete_sql))
            conn.commit()
            
            deleted_count += result.rowcount
            print(f"  - Deleted batch {i//batch_size_delete + 1}: {result.rowcount} rows")
        
        print(f"\n✓ Total rows deleted: {deleted_count}")
        
        # 5. Verification
        print("\n" + "-"*60)
        print("VERIFICATION")
        print("-"*60)
        
        # Verify deletion by checking remaining rows
        result = conn.execute(text("SELECT COUNT(*) FROM price"))
        remaining_count = result.fetchone()[0]
        
        # Check if any of the deleted names still exist (sample check)
        if names_list:
            sample_size = min(10, len(names_list))
            sample_names = names_list[:sample_size]
            quoted_sample = ["'" + name.replace("'", "''") + "'" for name in sample_names]
            check_sql = f"""
                SELECT COUNT(*) FROM price 
                WHERE name IN ({','.join(quoted_sample)})
            """
            result = conn.execute(text(check_sql))
            remaining_deleted = result.fetchone()[0]
        else:
            remaining_deleted = 0
        
        print(f"Expected remaining rows: {len(df) - len(df_to_delete)}")
        print(f"Actual remaining rows: {remaining_count}")
        print(f"Rows that should have been deleted but remain (sample): {remaining_deleted}")
        
        if remaining_count == len(df) - len(df_to_delete):
            print("✓ Verification successful!")
        else:
            print("⚠ Warning: Row count mismatch. Please check the backup file.")
        
        # 6. Summary report
        print("\n" + "="*80)
        print("DELETION SUMMARY")
        print("="*80)
        
        # Create summary DataFrame
        summary_data = []
        for name in list(names_to_delete)[:50]:  # Show first 50 for summary
            count = len(df_to_delete[df_to_delete['name'] == name])
            summary_data.append({'name': name, 'rows_deleted': count})
        
        df_summary = pd.DataFrame(summary_data)
        print("\nTop 50 deleted names:")
        print(df_summary.to_string(index=False))
        
        # Save summary report
        summary_file = os.path.join(dat_path, f"deletion_summary_{timestamp}.csv")
        df_summary.to_csv(summary_file, index=False)
        print(f"\n✓ Deletion summary saved to: {summary_file}")
        
        # 7. Rollback information
        print("\n" + "-"*60)
        print("ROLLBACK INFORMATION")
        print("-"*60)
        print(f"Backup file: {backup_file}")
        print(f"To restore deleted rows, you can run:")
        print(f"  df_restore = pd.read_csv('{backup_file}')")
        print(f"  df_restore.to_sql('price', con=const, if_exists='append', index=False)")
        
    except Exception as e:
        print(f"\n✗ Error during deletion: {e}")
        conn.rollback()
        print("Transaction rolled back. No changes were made.")
    
    finally:
        # Close connection
        conn.close()

print("\n" + "="*80)
print("OPERATION COMPLETED")
print("="*80)


DELETE ROWS FROM PRICE TABLE WHERE NAME NOT IN NAME LIST

------------------------------------------------------------
ANALYZING DATA
------------------------------------------------------------
Total unique names in price table: 233
Allowed names from df_name: 190
Names to delete: 43

------------------------------------------------------------
CREATING BACKUP OF ROWS TO BE DELETED
------------------------------------------------------------
Fetching rows to delete for 43 names...
  - Processed batch 1: found 50494 rows

✓ Total rows to delete: 50494
✓ Backup saved to: C:\Users\User\OneDrive\A4\Data\deleted_rows_backup_20260327_150841.csv

Sample of rows to delete (by name):
  - TIPCO: 1930 rows
  - KSL: 1809 rows
  - RS: 1783 rows
  - S11: 1779 rows
  - TK: 1778 rows
  - INTUCH: 1746 rows
  - EASTW: 1728 rows
  - ECL: 1721 rows
  - LHK: 1569 rows
  - STANLY: 1568 rows
  - PAP: 1567 rows
  - STEC: 1550 rows
  - SPRIME: 1494 rows
  - SKN: 1448 rows
  - AJ: 1442 rows
  - ESSO: 1418 row